# 🧠 English → SQL Converter
Converts natural language to SQL using GPT-4, executes against SQLite, and auto-debugs errors.

**Run cells top to bottom on first use.**

## ⚙️ Cell 1 — Install Dependencies

In [ ]:
# Run once to install required packages
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openai', 'pandas', 'tabulate', '-q'])
print('✅ Dependencies installed')

## 🗄️ Cell 2 — Create Sample Database

In [ ]:
import sqlite3, os

DB_PATH = 'sample.db'  # Change this path if needed

def create_sample_db(path=DB_PATH):
    conn = sqlite3.connect(path)
    cur = conn.cursor()

    cur.executescript("""
    CREATE TABLE IF NOT EXISTS departments (
        department_id INTEGER PRIMARY KEY,
        name          TEXT NOT NULL,
        budget        REAL
    );

    CREATE TABLE IF NOT EXISTS employees (
        employee_id   INTEGER PRIMARY KEY,
        name          TEXT NOT NULL,
        email         TEXT UNIQUE,
        department_id INTEGER REFERENCES departments(department_id),
        salary        REAL,
        hire_date     TEXT,
        role          TEXT
    );

    CREATE TABLE IF NOT EXISTS customers (
        customer_id INTEGER PRIMARY KEY,
        name        TEXT NOT NULL,
        email       TEXT UNIQUE,
        city        TEXT,
        signup_date TEXT
    );

    CREATE TABLE IF NOT EXISTS products (
        product_id INTEGER PRIMARY KEY,
        name       TEXT NOT NULL,
        category   TEXT,
        price      REAL,
        stock      INTEGER
    );

    CREATE TABLE IF NOT EXISTS orders (
        order_id    INTEGER PRIMARY KEY,
        customer_id INTEGER REFERENCES customers(customer_id),
        order_date  TEXT,
        status      TEXT,
        total       REAL
    );

    CREATE TABLE IF NOT EXISTS order_items (
        item_id    INTEGER PRIMARY KEY,
        order_id   INTEGER REFERENCES orders(order_id),
        product_id INTEGER REFERENCES products(product_id),
        quantity   INTEGER,
        unit_price REAL
    );
    """)

    cur.executemany('INSERT OR IGNORE INTO departments VALUES (?,?,?)', [
        (1,'Engineering',500000),(2,'Sales',300000),
        (3,'HR',150000),(4,'Marketing',200000)])

    cur.executemany('INSERT OR IGNORE INTO employees VALUES (?,?,?,?,?,?,?)', [
        (1,'Alice Chen','alice@co.com',1,120000,'2021-03-15','Senior Engineer'),
        (2,'Bob Smith','bob@co.com',2,85000,'2020-07-01','Sales Rep'),
        (3,'Carol Davis','carol@co.com',1,95000,'2022-01-10','Engineer'),
        (4,'Dan Lee','dan@co.com',3,70000,'2019-05-20','HR Manager'),
        (5,'Eva Kim','eva@co.com',4,80000,'2023-02-28','Marketing Lead'),
        (6,'Frank Wu','frank@co.com',2,90000,'2021-11-03','Sales Manager'),
        (7,'Grace Hall','grace@co.com',1,110000,'2020-09-14','Lead Engineer')])

    cur.executemany('INSERT OR IGNORE INTO customers VALUES (?,?,?,?,?)', [
        (1,'John Doe','john@email.com','New York','2023-01-15'),
        (2,'Jane Roe','jane@email.com','London','2022-08-20'),
        (3,'Mike Ray','mike@email.com','Tokyo','2023-05-10'),
        (4,'Sara Ali','sara@email.com','Paris','2021-12-01'),
        (5,'Tom Ng','tom@email.com','New York','2024-01-03')])

    cur.executemany('INSERT OR IGNORE INTO products VALUES (?,?,?,?,?)', [
        (1,'Laptop Pro','Electronics',1299.99,50),
        (2,'Wireless Mouse','Electronics',29.99,200),
        (3,'Standing Desk','Furniture',599.99,30),
        (4,'Notebook Set','Stationery',12.99,500),
        (5,'Coffee Maker','Appliances',89.99,75),
        (6,'Monitor 4K','Electronics',449.99,60)])

    cur.executemany('INSERT OR IGNORE INTO orders VALUES (?,?,?,?,?)', [
        (1,1,'2024-01-10','completed',1329.98),
        (2,2,'2024-01-15','completed',599.99),
        (3,3,'2024-02-01','pending',449.99),
        (4,1,'2024-02-14','completed',89.99),
        (5,4,'2024-03-01','cancelled',12.99),
        (6,5,'2024-03-15','completed',1749.98)])

    cur.executemany('INSERT OR IGNORE INTO order_items VALUES (?,?,?,?,?)', [
        (1,1,1,1,1299.99),(2,1,2,1,29.99),
        (3,2,3,1,599.99),
        (4,3,6,1,449.99),
        (5,4,5,1,89.99),
        (6,5,4,1,12.99),
        (7,6,1,1,1299.99),(8,6,3,1,449.99)])

    conn.commit()
    conn.close()
    print(f'✅ Database created: {os.path.abspath(path)}')

create_sample_db()

# Print schema for reference
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = [r[0] for r in cur.fetchall()]
print('\n📋 Schema:')
for t in tables:
    cur.execute(f'PRAGMA table_info({t})')
    cols = [f"{r[1]} ({r[2]})" for r in cur.fetchall()]
    print(f'  {t}: {", ".join(cols)}')
conn.close()

## 🔑 Cell 3 — Configure OpenAI API Key

In [ ]:
import os

# Option A: Set inline (not recommended for shared notebooks)
# os.environ['OPENAI_API_KEY'] = 'sk-...'

# Option B: Read from environment (recommended)
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

if not OPENAI_API_KEY:
    # Prompt interactively
    import getpass
    OPENAI_API_KEY = getpass.getpass('Enter your OpenAI API key: ')
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

print('✅ API key set' if OPENAI_API_KEY else '❌ No API key provided')

## 🧠 Cell 4 — Core Engine

In [ ]:
import sqlite3, json, re, os
import pandas as pd
from openai import OpenAI

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
MAX_RETRIES = 3

# ── Schema extractor ──────────────────────────────────────────────────────────

def get_schema(db_path: str) -> str:
    """Return a compact DDL-style schema string for the GPT system prompt."""
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
    tables = [r[0] for r in cur.fetchall()]
    lines = []
    for t in tables:
        cur.execute(f'PRAGMA table_info({t})')
        cols = [f"{r[1]} {r[2]}" for r in cur.fetchall()]
        lines.append(f"{t}({', '.join(cols)})")
    conn.close()
    return '\n'.join(lines)

# ── SQL generator ─────────────────────────────────────────────────────────────

def generate_sql(english: str, schema: str, error_context: str = '') -> str:
    """Call GPT-4 to produce a SQL query from an English question."""
    system = f"""You are a SQLite SQL expert. Convert the user's English question into a single executable SQLite SQL query.

Database schema:
{schema}

Rules:
- Return ONLY the raw SQL statement, no markdown, no explanation, no code fences.
- Use correct SQLite syntax (no ILIKE, use LIKE; no NOW(), use date('now')).
- Never use DROP, DELETE, or ALTER unless the user explicitly confirms.
"""
    if error_context:
        system += f"\nPrevious attempt failed with: {error_context}\nFix the SQL accordingly."

    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': english}
        ],
        temperature=0
    )
    sql = response.choices[0].message.content.strip()
    # Strip markdown fences if GPT wraps anyway
    sql = re.sub(r'^```[\w]*\n?', '', sql)
    sql = re.sub(r'\n?```$', '', sql)
    return sql.strip()

# ── SQL executor ──────────────────────────────────────────────────────────────

def execute_sql(sql: str, db_path: str):
    """Run SQL and return (rows, columns, error). rows=None on error."""
    try:
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description] if cur.description else []
        conn.commit()
        conn.close()
        return rows, cols, None
    except Exception as e:
        return None, None, str(e)

# ── Main converter ────────────────────────────────────────────────────────────

def english_to_sql(english: str, db_path: str = DB_PATH) -> dict:
    """
    Convert English → SQL → execute → auto-debug.
    Returns a result dict with all attempt details.
    """
    schema = get_schema(db_path)
    result = {
        'english_query':    english,
        'sql_generated':    None,
        'execution_status': None,
        'error_message':    None,
        'debug_attempts':   [],
        'final_sql':        None,
        'final_status':     None,
        'result_rows':      None,
        'dataframe':        None
    }

    # Initial generation
    sql = generate_sql(english, schema)
    result['sql_generated'] = sql

    rows, cols, error = execute_sql(sql, db_path)

    if not error:
        result.update({
            'execution_status': 'success',
            'final_sql':        sql,
            'final_status':     'success',
            'result_rows':      len(rows),
            'dataframe':        pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame(columns=cols)
        })
        return result

    # Failed — enter debug loop
    result['execution_status'] = 'failed'
    result['error_message']    = error

    for attempt in range(1, MAX_RETRIES + 1):
        print(f'  🔧 Debug attempt {attempt}/{MAX_RETRIES}: {error}')
        sql = generate_sql(english, schema, error_context=f"{error}\nFailed SQL:\n{sql}")
        rows, cols, error = execute_sql(sql, db_path)

        entry = {'attempt': attempt, 'modified_sql': sql, 'error': error}
        if not error:
            entry['status'] = 'success'
            result['debug_attempts'].append(entry)
            result.update({
                'final_sql':    sql,
                'final_status': 'success',
                'result_rows':  len(rows),
                'dataframe':    pd.DataFrame(rows, columns=cols) if rows else pd.DataFrame(columns=cols)
            })
            return result
        else:
            entry['status'] = 'failed'
            result['debug_attempts'].append(entry)

    result.update({
        'final_sql':    sql,
        'final_status': f'failed_after_{MAX_RETRIES}_attempts'
    })
    return result

# ── Pretty printer ────────────────────────────────────────────────────────────

def display_result(result: dict):
    print('=' * 60)
    print(f'📝 Query   : {result["english_query"]}')
    print(f'🗃️  SQL     : {result["sql_generated"]}')
    print(f'📊 Status  : {result["final_status"]}')

    if result['debug_attempts']:
        print(f'🔧 Debugged: {len(result["debug_attempts"])} attempt(s)')
        if result['final_sql'] != result['sql_generated']:
            print(f'✏️  Final SQL: {result["final_sql"]}')

    if result['final_status'] == 'success':
        print(f'✅ Rows    : {result["result_rows"]}')
        print()
        display(result['dataframe'])
    else:
        print(f'❌ Error   : {result["error_message"]}')
    print('=' * 60)

    # Also return JSON summary (without dataframe)
    summary = {k: v for k, v in result.items() if k != 'dataframe'}
    return summary

print('✅ Engine loaded — ready to query!')

## 🚀 Cell 5 — Run Your Query
Change the `QUERY` string below to anything you like.

In [ ]:
QUERY = "Show me all customers from New York"

result = english_to_sql(QUERY)
summary = display_result(result)

## 📦 Cell 6 — JSON Output
Full structured result as required by the spec.

In [ ]:
import json
print(json.dumps(summary, indent=2))

## 🔁 Cell 7 — Batch Queries

In [ ]:
queries = [
    "List all employees in the Engineering department",
    "What are the top 3 most expensive products?",
    "Show total revenue per customer, sorted highest first",
    "How many orders were completed vs pending vs cancelled?",
    "Which employees earn more than the average salary?",
]

for q in queries:
    r = english_to_sql(q)
    display_result(r)
    print()

## 🛠️ Cell 8 — Use Your Own Database
Point the engine at any SQLite `.db` file.

In [ ]:
# YOUR_DB = '/path/to/your/database.db'
# result = english_to_sql('Your question here', db_path=YOUR_DB)
# display_result(result)